In [2]:
import pandas as pd

# Nombre del archivo combinado que queremos diagnosticar
nombre_archivo_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\combined_training_set.csv"

try:
    print(f"--- DIAGNÓSTICO DEL ARCHIVO: '{nombre_archivo_csv}' ---")
    
    # Cargar el archivo, intentando con punto y coma primero
    try:
        df = pd.read_csv(nombre_archivo_csv, sep=';')
        if df.shape[1] == 1:
             df = pd.read_csv(nombre_archivo_csv, sep=',')
    except:
        df = pd.read_csv(nombre_archivo_csv, sep=',')

    # La parte clave: Contar los valores en la columna 'Actividad'
    if 'Actividad' in df.columns:
        print("\nConteo de valores en la columna 'Actividad':")
        print(df['Actividad'].value_counts())
    else:
        print("\n❌ ERROR: No se encontró la columna 'Actividad' en el archivo.")

except FileNotFoundError:
    print(f"❌ ERROR: No se pudo encontrar el archivo '{nombre_archivo_csv}'.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")

--- DIAGNÓSTICO DEL ARCHIVO: 'C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\combined_training_set.csv' ---

Conteo de valores en la columna 'Actividad':
Actividad
Act1     2332
Act-1    2025
Name: count, dtype: int64


In [8]:
import pandas as pd
import sys

# --- Configuración ---
ruta_dm_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\bases de dato\training_dmm.csv"
ruta_fp_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos Fingerprint\Bases de datos\new_training.csv"
nombre_salida_csv = "combined_training_set.csv"

try:
    print("--- PASO 1: Cargando archivos... ---")
    df_dm = pd.read_csv(ruta_dm_csv, sep=';')
    df_fp = pd.read_csv(ruta_fp_csv, sep=',')

    # --- PASO 2: Limpieza de columnas ---
    print("--- PASO 2: Limpiando columnas innecesarias... ---")
    
    # ¡NUEVO! Eliminamos la columna 'Serie' del dataframe de fingerprints
    if 'Serie' in df_fp.columns:
        df_fp = df_fp.drop(columns=['Serie'])
        print("  > Columna 'Serie' eliminada exitosamente.")
    
    # Verificación de consistencia
    if len(df_dm) != len(df_fp):
        print("\n❌ ¡ERROR CRÍTICO! El número de filas no coincide.")
        sys.exit()

    print("\n--- PASO 3: Combinando los datasets... ---")
    df_dm_features = df_dm.drop(columns=['Actividad'], errors='ignore')
    df_fp_features = df_fp.drop(columns=['Actividad', 'Name', 'Set'], errors='ignore')
    df_actividad = df_fp[['Actividad']]
    
    df_combined_features = pd.concat([df_dm_features.reset_index(drop=True), df_fp_features.reset_index(drop=True)], axis=1)
    df_combined_final = pd.concat([df_combined_features, df_actividad], axis=1)
    
    # Guardar el archivo final
    df_combined_final.to_csv(nombre_salida_csv, index=False, sep=';')
    
    print(f"\n✅ ¡Éxito! Archivo '{nombre_salida_csv}' guardado sin la columna 'Serie'.")

except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")

--- PASO 1: Cargando archivos... ---
--- PASO 2: Limpiando columnas innecesarias... ---
  > Columna 'Serie' eliminada exitosamente.

--- PASO 3: Combinando los datasets... ---

✅ ¡Éxito! Archivo 'combined_training_set.csv' guardado sin la columna 'Serie'.


In [9]:
import pandas as pd
import sys

# =============================================================================
#  CONFIGURACIÓN
# =============================================================================
# Archivos de entrada
ruta_dm_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\bases de dato\training_dmm.csv"
ruta_fp_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos Fingerprint\Bases de datos\new_training.csv"

# Archivo de salida final
nombre_archivo_arff = "combined_training_final.arff"
nombre_relacion = "pampa_combined_dataset"
# =============================================================================

try:
    # --- PASO 1: Cargar los datasets originales ---
    print("--- PASO 1: Cargando archivos originales... ---")
    
    # Cargar DMs (sabemos que puede usar ';')
    df_dm = pd.read_csv(ruta_dm_csv, sep=';')
    print(f"  > Archivo DM cargado. Columnas detectadas: {df_dm.shape[1]}")
    if df_dm.shape[1] < 2:
        print("    > Reintentando con separador ','...")
        df_dm = pd.read_csv(ruta_dm_csv, sep=',')
        print(f"    > Archivo DM cargado. Columnas detectadas: {df_dm.shape[1]}")

    # Cargar FPs (usa ',')
    df_fp = pd.read_csv(ruta_fp_csv, sep=',')
    print(f"  > Archivo FP cargado. Columnas detectadas: {df_fp.shape[1]}")
    
    if len(df_dm) != len(df_fp):
        print("\n❌ ¡ERROR CRÍTICO! El número de filas no coincide.")
        sys.exit()

    # --- PASO 2: Combinar los datos en memoria ---
    print("\n--- PASO 2: Combinando los datasets... ---")
    df_dm_features = df_dm.drop(columns=['Actividad'], errors='ignore')
    df_fp_features = df_fp.drop(columns=['Actividad', 'Name', 'Set', 'Serie'], errors='ignore')
    df_actividad = df_fp[['Actividad']]
    
    df_combined = pd.concat([
        df_dm_features.reset_index(drop=True), 
        df_fp_features.reset_index(drop=True),
        df_actividad.reset_index(drop=True)
    ], axis=1)
    print("  > Fusión completada.")

    # --- PASO 3: Escribir el archivo ARFF directamente ---
    print("\n--- PASO 3: Escribiendo el archivo ARFF final... ---")
    with open(nombre_archivo_arff, 'w', encoding='utf-8') as f:
        f.write(f"@relation {nombre_relacion}\n\n")
        
        for column in df_combined.columns:
            # Limpiar nombres de columna para máxima compatibilidad
            clean_column = f"'{str(column).replace(' ', '_')}'"
            
            if df_combined[column].dtype in ['int64', 'float64']:
                f.write(f"@attribute {clean_column} numeric\n")
            else:
                unique_values = df_combined[column].unique()
                unique_values_str = ",".join([str(val).replace("'", "") for val in unique_values])
                f.write(f"@attribute {clean_column} {{{unique_values_str}}}\n")
        
        f.write("\n@data\n")
        
        # Guardar los datos usando el formato CSV estándar (comas) que ARFF espera
        csv_data = df_combined.to_csv(index=False, header=False)
        f.write(csv_data)

    print(f"\n✅ ¡Éxito! Archivo '{nombre_archivo_arff}' creado correctamente.")
    print("   Este archivo está listo para ser usado en WEKA sin errores de formato.")

except FileNotFoundError as e:
    print(f"❌ ERROR: No se pudo encontrar el archivo: {e.filename}")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")

--- PASO 1: Cargando archivos originales... ---
  > Archivo DM cargado. Columnas detectadas: 1
    > Reintentando con separador ','...
    > Archivo DM cargado. Columnas detectadas: 618
  > Archivo FP cargado. Columnas detectadas: 168

--- PASO 2: Combinando los datasets... ---
  > Fusión completada.

--- PASO 3: Escribiendo el archivo ARFF final... ---

✅ ¡Éxito! Archivo 'combined_training_final.arff' creado correctamente.
   Este archivo está listo para ser usado en WEKA sin errores de formato.
